In [4]:
# ===============================
# CONFIG (여기만 수정하세요)
# ===============================
# 날짜 목록: YYYYMMDDHH 형식
date_list = [
    '2012072700',
    '2017072400',
    '2022082800',
]

n_ens              = 100   # 앙상블 수 (0번은 control run)
perturbation_scale = 0.05  # 노이즈 크기 (std 대비 비율)
factor_list        = ['z'] # perturb할 변수: 'z','q','t','u','v','MSLP','U10','V10','T2M'
forecast_steps     = 20    # 예측 스텝 수 (×6h, 20 = 120h)
gpu_device_id      = 1     # CUDA device ID

# 저장 영역 (경도 0~360 기준)
lon_min, lon_max = 90, 180  # E
lat_min, lat_max =  0,  60  # N

# 경로
era5_base   = '/data06/ERA5/hourly'
pangu_dir   = '/home1/jek/Pangu-Weather'
output_base = '/data09/Pangu_TC_ENS/output_data'

In [2]:
import torch
import onnxruntime as ort
import netCDF4 as nc
import numpy as np
import os
import time
from datetime import datetime, timezone

# GPU 확인
if torch.cuda.is_available():
    print(f'Using GPU {gpu_device_id}: {torch.cuda.get_device_name(gpu_device_id)}')
else:
    print('WARNING: CUDA not available')

# 저장 영역 인덱스 (날짜와 무관하게 고정)
lat_grid = np.linspace(90, -90, 721)
lon_grid = np.linspace(0, 359.75, 1440)
lat_start = int(np.argmin(np.abs(lat_grid - lat_max)))
lat_end   = int(np.argmin(np.abs(lat_grid - lat_min)))
lon_start = int(np.argmin(np.abs(lon_grid - lon_min)))
lon_end   = int(np.argmin(np.abs(lon_grid - lon_max)))
print(f'Subregion: lat {lat_max}N→{lat_min}N  idx={lat_start}:{lat_end+1}')
print(f'           lon {lon_min}E→{lon_max}E  idx={lon_start}:{lon_end+1}')

# Pangu 기압면 (13개)
PANGU_LEVELS = [1000, 925, 850, 700, 600, 500, 400, 300, 250, 200, 150, 100, 50]

# 변수 dict
surface_dict = {'MSLP': 0, 'U10': 1, 'V10': 2, 'T2M': 3}
upper_dict   = {'z': 0, 'q': 1, 't': 2, 'u': 3, 'v': 4}

# ONNX 세션 (한 번만 로드)
options = ort.SessionOptions()
options.enable_cpu_mem_arena = True
options.enable_mem_pattern   = False
options.enable_mem_reuse     = False
cuda_opts = {'arena_extend_strategy': 'kSameAsRequested', 'device_id': gpu_device_id}
ort_session_6 = ort.InferenceSession(
    f'{pangu_dir}/pangu_weather_6.onnx',
    sess_options=options,
    providers=[('CUDAExecutionProvider', cuda_opts)]
)
print(f'Loaded pangu_weather_6.onnx on GPU {gpu_device_id}')

# ERA5 로드 함수
def load_upper_var(var_name, yyyymm, target_ts):
    """상층 변수 1개 → Pangu 13개 기압면 선택. shape: (13, 721, 1440)"""
    ds = nc.Dataset(f'{era5_base}/{var_name}/ERA5_{var_name}_{yyyymm}.nc')
    t_idx = int(np.argmin(np.abs(ds.variables['valid_time'][:] - target_ts)))
    levels = ds.variables['pressure_level'][:]
    lev_idx = [int(np.argmin(np.abs(levels - lv))) for lv in PANGU_LEVELS]
    data = np.array(ds.variables[var_name][t_idx, lev_idx, :, :], dtype=np.float32)
    ds.close()
    return data

def load_surface_var(nc_dir, var_name, yyyymm, target_ts):
    """지면 변수 1개. shape: (721, 1440)"""
    ds = nc.Dataset(f'{era5_base}/{nc_dir}/ERA5_{nc_dir}_{yyyymm}.nc')
    t_idx = int(np.argmin(np.abs(ds.variables['valid_time'][:] - target_ts)))
    data = np.array(ds.variables[var_name][t_idx, :, :], dtype=np.float32)
    ds.close()
    return data

factor_str = ''.join([f'_{f}' for f in factor_list])
print(f'\nReady. {len(date_list)} dates × {n_ens} ens × {forecast_steps} steps')

Using GPU 1: NVIDIA A40
Subregion: lat 60N→0N  idx=120:361
           lon 90E→180E  idx=360:721
Loaded pangu_weather_6.onnx on GPU 1

Ready. 5 dates × 100 ens × 20 steps


In [5]:
from tqdm import tqdm

# ===============================
# 날짜 × 앙상블 메인 루프
# ===============================
all_start = time.time()

for date_str in date_list:
    # --- 날짜 파싱 ---
    dt = datetime.strptime(date_str, '%Y%m%d%H').replace(tzinfo=timezone.utc)
    yyyymm   = dt.strftime('%Y%m')
    time_str = dt.strftime('%Y/%m/%d/%HUTC')
    target_ts = dt.timestamp()

    print(f'\n=== {time_str} ===')

    # --- ERA5 로드 ---
    t0 = time.time()
    upper_list = [load_upper_var(v, yyyymm, target_ts) for v in ['z', 'q', 't', 'u', 'v']]
    input_upper = np.stack(upper_list, axis=0)  # (5, 13, 721, 1440)

    sfc_list = [load_surface_var(nc_dir, var, yyyymm, target_ts)
                for nc_dir, var in [('msl','msl'),('10u','u10'),('10v','v10'),('2t','t2m')]]
    input_surface = np.stack(sfc_list, axis=0)  # (4, 721, 1440)
    print(f'ERA5 loaded [{time.time()-t0:.1f}s]')

    # 표준편차 계산 (perturbation 기준)
    std_dev_upper   = np.std(input_upper,   axis=(2, 3), dtype=np.float32) * perturbation_scale
    std_dev_surface = np.std(input_surface, axis=(1, 2), dtype=np.float32) * perturbation_scale

    ens_root = f'{output_base}/{time_str}/{perturbation_scale}ENS{factor_str}_6h'

    # --- 앙상블 루프 ---
    for ens in tqdm(range(n_ens), desc='ENSEMBLE'):
        if date_str == '2012072700' and ens <= 40:
            print(f'  Skipping ENS{ens:03d} for {date_str} (already done)')
            continue
        out_dir = os.path.join(ens_root, str(ens))
        os.makedirs(os.path.join(out_dir, 'upper'),   exist_ok=True)
        os.makedirs(os.path.join(out_dir, 'surface'), exist_ok=True)

        # Perturbation
        perturbed_upper   = input_upper.copy()
        perturbed_surface = input_surface.copy()

        if ens != 0:
            seed_val = hash((ens, tuple(factor_list), perturbation_scale)) % (2**32)
            rng = np.random.default_rng(seed_val)
            for factor in factor_list:
                if factor in upper_dict:
                    idx = upper_dict[factor]
                    for j in range(13):
                        noise = rng.normal(0, std_dev_upper[idx, j],
                                           input_upper[idx, j].shape).astype(np.float32)
                        perturbed_upper[idx, j] = input_upper[idx, j] + noise
                elif factor in surface_dict:
                    idx = surface_dict[factor]
                    noise = rng.normal(0, std_dev_surface[idx],
                                       input_surface[idx].shape).astype(np.float32)
                    perturbed_surface[idx] = input_surface[idx] + noise

        # 0h 저장
        np.save(os.path.join(out_dir, 'upper/0h'),
                perturbed_upper[:, :, lat_start:lat_end+1, lon_start:lon_end+1])
        np.save(os.path.join(out_dir, 'surface/0h'),
                perturbed_surface[:, lat_start:lat_end+1, lon_start:lon_end+1])

        # 6h 반복 예측
        cur_upper, cur_surface = perturbed_upper, perturbed_surface
        ens_start = time.time()

        for i in range(forecast_steps):
            lead = 6 * (i + 1)
            step_t = time.time()
            cur_upper, cur_surface = ort_session_6.run(
                None, {'input': cur_upper, 'input_surface': cur_surface}
            )
            np.save(os.path.join(out_dir, f'upper/{lead}h'),
                    cur_upper[:, :, lat_start:lat_end+1, lon_start:lon_end+1])
            np.save(os.path.join(out_dir, f'surface/{lead}h'),
                    cur_surface[:, lat_start:lat_end+1, lon_start:lon_end+1])
            # print(f'  {time_str} ENS{ens:03d} +{lead:3d}h  [{time.time()-step_t:.1f}s]')

        # print(f'  ENS{ens:03d} done: {time.time()-ens_start:.1f}s')

print(f'\n>>> All dates done: {time.time()-all_start:.1f}s')


=== 2012/07/27/00UTC ===
ERA5 loaded [25.5s]


ENSEMBLE:   0%|          | 0/100 [00:00<?, ?it/s]

  Skipping ENS000 for 2012072700 (already done)
  Skipping ENS001 for 2012072700 (already done)
  Skipping ENS002 for 2012072700 (already done)
  Skipping ENS003 for 2012072700 (already done)
  Skipping ENS004 for 2012072700 (already done)
  Skipping ENS005 for 2012072700 (already done)
  Skipping ENS006 for 2012072700 (already done)
  Skipping ENS007 for 2012072700 (already done)
  Skipping ENS008 for 2012072700 (already done)
  Skipping ENS009 for 2012072700 (already done)
  Skipping ENS010 for 2012072700 (already done)
  Skipping ENS011 for 2012072700 (already done)
  Skipping ENS012 for 2012072700 (already done)
  Skipping ENS013 for 2012072700 (already done)
  Skipping ENS014 for 2012072700 (already done)
  Skipping ENS015 for 2012072700 (already done)
  Skipping ENS016 for 2012072700 (already done)
  Skipping ENS017 for 2012072700 (already done)
  Skipping ENS018 for 2012072700 (already done)
  Skipping ENS019 for 2012072700 (already done)
  Skipping ENS020 for 2012072700 (alread

ENSEMBLE: 100%|██████████| 100/100 [46:33<00:00, 27.94s/it]



=== 2017/07/24/00UTC ===
ERA5 loaded [21.6s]


ENSEMBLE: 100%|██████████| 100/100 [1:24:52<00:00, 50.93s/it]



=== 2022/08/28/00UTC ===
ERA5 loaded [22.6s]


ENSEMBLE: 100%|██████████| 100/100 [1:19:25<00:00, 47.65s/it]


>>> All dates done: 12722.0s
